# Modelo con atención: cinco stems a MIDI
Demucs separa `vocals`, `drums`, `bass`, `guitar`, `piano` y `other` una vez por mezcla. Un único transcriptor con pesos compartidos y condicionado por stem procesa voz, bajo, guitarra y piano; batería usa un segundo modelo de golpes. La separación conserva la identidad y reduce la interferencia entre instrumentos. El transcriptor compartido combina bloques residuales, atención por pitch y tiempo, y una compuerta Mel/croma aprendida. Adjuntá [GuitarSet](https://www.kaggle.com/datasets/alonhaviv/guitarset) y [Slakh2100](https://www.kaggle.com/datasets/alonhaviv/slakh2100) con **Add Input**. El notebook lee ambos desde `/kaggle/input/datasets/alonhaviv/`. `AUTO_DOWNLOAD_VOCADITO = True` descarga solo [Vocadito v3](https://zenodo.org/records/5578807) desde Zenodo. Slakh2100-redux conserva sus particiones oficiales y excluye los temas duplicados de `omitted`.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
REPO = Path('/kaggle/working/mia-final-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Kajachuan/mia-final-project.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## Entradas y reanudación
Adjuntá la salida anterior como **Notebook Output Files** y completá `PREVIOUS_OUTPUT` para continuar. `TOTAL_EPOCHS` es el objetivo total acumulado. Las descargas auxiliares y las características se omiten si la caché adjunta es compatible.

In [ ]:
GUITARSET_ROOT = next((p for p in (Path('/kaggle/input/datasets/alonhaviv/guitarset'), Path('/kaggle/input/guitarset')) if p.is_dir()), Path('/kaggle/input/datasets/alonhaviv/guitarset'))
SLAKH_ROOT = next((p for p in (Path('/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux'), Path('/kaggle/input/slakh2100/slakh2100_flac_redux')) if p.is_dir()), Path('/kaggle/input/datasets/alonhaviv/slakh2100/slakh2100_flac_redux'))
VOCADITO_ROOT = None   # Opcional: carpeta de Vocadito ya adjuntada
PREVIOUS_OUTPUT = None # Ej.: '/kaggle/input/mi-notebook-anterior'
AUTO_DOWNLOAD_VOCADITO = True
REBUILD_AUX_CACHE = False  # True para forzar el reprocesamiento
MAX_SLAKH_TRACKS = 100  # Subconjunto de Slakh2100; 0 procesa todos los temas
STEPS_PER_STEM = 100  # Cada época toma este número de lotes de cada stem con altura
TOTAL_EPOCHS = 12
RUN_BUDGET_MINUTES = 180
CHECKPOINT_EVERY_BATCHES = 25
WORK = Path('/kaggle/working')
RUNS = WORK / 'runs_five'
RUNS.mkdir(exist_ok=True)
previous = Path(PREVIOUS_OUTPUT) if PREVIOUS_OUTPUT else None
if not GUITARSET_ROOT.is_dir():
    raise FileNotFoundError('Adjuntá alonhaviv/guitarset con Add Input')
if not SLAKH_ROOT.is_dir():
    raise FileNotFoundError('Adjuntá alonhaviv/slakh2100 con Add Input o ajustá SLAKH_ROOT')
if not all((SLAKH_ROOT / split).is_dir() for split in ('train', 'validation', 'test')):
    raise FileNotFoundError(f'Slakh debe contener train, validation y test: {SLAKH_ROOT}')
if previous and not previous.is_dir():
    raise FileNotFoundError(previous)
STARTED = time.monotonic()

In [ ]:
from polytranscribe.features import FEATURE_VERSION
def compatible(path, stem=None):
    manifest = path / 'manifest.json'
    if not manifest.exists():
        return False
    info = json.loads(manifest.read_text())
    return info.get('feature_version') == FEATURE_VERSION and (stem not in ('bass', 'piano', 'drums') or (info.get('source') == str(SLAKH_ROOT) and info.get('max_slakh_tracks') == MAX_SLAKH_TRACKS))
GUITAR_CACHE = previous / 'guitarset/cache' if previous and compatible(previous / 'guitarset/cache') else WORK / 'guitarset/cache'
if not compatible(GUITAR_CACHE):
    subprocess.run([sys.executable, '-m', 'polytranscribe.prepare', '--source-root', str(GUITARSET_ROOT),
                    '--root', str(WORK / 'guitarset')], check=True)
    GUITAR_CACHE = WORK / 'guitarset/cache'
AUX_CACHE = previous / 'five_cache' if previous and not REBUILD_AUX_CACHE and all(compatible(previous / 'five_cache' / s, s) for s in ('vocals','bass','piano','drums')) else WORK / 'five_cache'
if REBUILD_AUX_CACHE or not all(compatible(AUX_CACHE / s, s) for s in ('vocals','bass','piano','drums')):
    aux = WORK / 'aux'
    if VOCADITO_ROOT is None:
        old = previous / 'aux/vocadito' if previous else None
        if old and (old / '.ready.json').exists():
            VOCADITO_ROOT = old
        elif AUTO_DOWNLOAD_VOCADITO:
            from polytranscribe.download_aux import fetch
            VOCADITO_ROOT = fetch('vocadito', aux)
    if VOCADITO_ROOT is None:
        raise FileNotFoundError('Adjuntá Vocadito o activá AUTO_DOWNLOAD_VOCADITO')
    subprocess.run([sys.executable, '-m', 'polytranscribe.prepare_five',
                    '--slakh-root', str(SLAKH_ROOT), '--vocadito-root', str(VOCADITO_ROOT),
                    '--output', str(WORK / 'five_cache'), '--max-slakh-tracks', str(MAX_SLAKH_TRACKS)], check=True)
    AUX_CACHE = WORK / 'five_cache'
CACHES = {'guitar': GUITAR_CACHE, **{s: AUX_CACHE / s for s in ('vocals','bass','piano','drums')}}
print('Cachés:', CACHES)

## Entrenar y guardar
Se entrena `shared_gated` para los cuatro stems con altura y `advanced` para batería. `shared_plain` permite aislar el aporte de la compuerta con exactamente el mismo entrenamiento balanceado. El primer modelo conserva un solo conjunto de pesos y umbrales de validación por stem. El notebook guarda modelo, optimizador y paso cada 25 lotes para continuar desde una versión anterior. Los modelos separados de guitarra siguen disponibles como controles en el notebook de GuitarSet.

In [ ]:
import torch
from polytranscribe.train import CHECKPOINT_FORMAT, manifest_identity
from polytranscribe.train_shared import STEMS, shared_cache_identity
def compatible_resume(path, variant):
    if not path.exists():
        return False
    saved = torch.load(path, map_location='cpu', weights_only=False)
    if variant == 'advanced':
        manifest = json.loads((CACHES['drums'] / 'manifest.json').read_text())
        identity = manifest_identity(manifest)
    else:
        manifests = {stem: json.loads((CACHES[stem] / 'manifest.json').read_text()) for stem in STEMS}
        identity = shared_cache_identity(manifests, 4, 256, STEPS_PER_STEM, 3e-4)
    return saved.get('feature_version') == FEATURE_VERSION and saved.get('checkpoint_format') == CHECKPOINT_FORMAT and saved.get('cache_identity') == identity and saved.get('variant') == variant
JOBS = ('shared_gated', 'advanced', 'shared_plain')
deadline = STARTED + RUN_BUDGET_MINUTES * 60
for variant in JOBS:
    remaining = (deadline - time.monotonic()) / 60 - 5
    if remaining <= 0:
        print('Presupuesto agotado: guardá esta versión y continuá después')
        break
    output = RUNS / 'drums' if variant == 'advanced' else RUNS
    output.mkdir(exist_ok=True)
    candidates = [output / f'{variant}_last.pt']
    if previous:
        candidates.append(previous / 'runs_five' / ('drums' if variant == 'advanced' else '') / f'{variant}_last.pt')
    resume = next((path for path in candidates if compatible_resume(path, variant)), None)
    if variant == 'advanced':
        command = [sys.executable, '-m', 'polytranscribe.train', '--cache', str(CACHES['drums'])]
    else:
        command = [sys.executable, '-m', 'polytranscribe.train_shared', '--guitar-cache', str(CACHES['guitar']), '--aux-cache', str(AUX_CACHE), '--steps-per-stem', str(STEPS_PER_STEM)]
    command += ['--output', str(output), '--variant', variant, '--epochs', str(TOTAL_EPOCHS),
                '--batch-size', '4', '--window-frames', '256',
                '--checkpoint-every', str(CHECKPOINT_EVERY_BATCHES),
                '--max-runtime-minutes', str(remaining)]
    if resume:
        command += ['--resume', str(resume)]
    print('Entrenando', variant, '| resume:', resume, flush=True)
    subprocess.run(command, check=True)
    status = json.loads((output / f'{variant}_status.json').read_text())
    print(variant, status, flush=True)
    if status['status'] != 'complete':
        break

In [ ]:
for variant in JOBS:
    result = (RUNS / 'drums' if variant == 'advanced' else RUNS) / f'{variant}.json'
    if result.exists():
        scores = json.loads(result.read_text())['test']
        if variant == 'advanced':
            print('drums', 'note F1:', scores['note']['f1'])
        else:
            print(variant, {stem: scores[stem]['note']['f1'] for stem in STEMS})

In [ ]:
MIXTURE_AUDIO = None  # Ejemplo: '/kaggle/input/mi-mezcla/cancion.wav'
ready = all(path.exists() and status.exists() and json.loads(status.read_text())['status'] == 'complete'
            for path, status in ((RUNS / 'shared_gated.pt', RUNS / 'shared_gated_status.json'),
                                 (RUNS / 'drums/advanced.pt', RUNS / 'drums/advanced_status.json')))
if ready and time.monotonic() < deadline - 60:
    if MIXTURE_AUDIO:
        audio = Path(MIXTURE_AUDIO)
    elif (SLAKH_ROOT / 'test').is_dir():
        audio = next((SLAKH_ROOT / 'test').rglob('mix.flac'))
    else:
        audio = next(GUITARSET_ROOT.rglob('*_mic.wav'))
    subprocess.run([sys.executable, '-m', 'polytranscribe.pipeline', '--audio', str(audio),
                    '--models-root', str(RUNS), '--output-dir', str(RUNS / 'demo')], check=True)
else:
    print('Faltan checkpoints o tiempo para la prueba de cinco stems')

## Comparación externa opcional
Para comparar con Basic Pitch u otro modelo preentrenado, generá MIDI para los 60 fragmentos de prueba de GuitarSet (intérprete 05), con un archivo `{stem}.mid` por fragmento. Adjuntá esos MIDI como entrada y fijá `EXTERNAL_MIDI_DIR`. El evaluador usa exactamente las mismas tolerancias de `mir_eval`; no entrena ni descarga el modelo externo.

In [ ]:
EXTERNAL_MIDI_DIR = None  # Ejemplo: '/kaggle/input/basic-pitch-guitarset-test'
if EXTERNAL_MIDI_DIR:
    subprocess.run([sys.executable, '-m', 'polytranscribe.benchmark_external',
                    '--guitarset-root', str(GUITARSET_ROOT),
                    '--predictions', str(EXTERNAL_MIDI_DIR),
                    '--output', str(RUNS / 'external_baseline.json')], check=True)

## Interpretación de resultados
Los F1 de GuitarSet, Vocadito y Slakh miden el reconocimiento en stems aislados. Para evaluar mezcla → cinco pistas se necesita un conjunto con referencia de voz además de las cuatro familias de Slakh; BabySlakh no contiene voz. Esta primera versión verifica la salida de cinco pistas pero **no declara una precisión de cinco instrumentos sobre mezclas reales**. Guardá la versión del notebook antes del límite de sesión para persistir `runs_five/` y `five_cache/`.